In [3]:
import pandas as pd
import numpy as np
import statsmodels.formula.api as smf
from sklearn.preprocessing import StandardScaler

# Load the cleaned dataset
df = pd.read_csv("/Users/kimballweeks/Downloads/cleaned_data.csv")

# Rename for compatibility
df = df.rename(columns={
    "pct_highschool_or_more (1990)": "pct_hs_1990",
    "Pop 2023": "pop_2023",
    "Established firms 2022": "firms_2022",
    "Established firms 1989": "firms_1989"
})

# Convert relevant columns to numeric
cols_to_numeric = [
    'church_persistence_index',
    'church_density_2020',
    'income_1989_real_2023',
    'pct_hs_1990',
    'pop_2023',
    'firms_2022',
    'firms_1989'
]
for col in cols_to_numeric:
    df[col] = pd.to_numeric(df[col], errors='coerce')

# Drop rows with missing values
df = df.dropna(subset=cols_to_numeric)

# Drop rows where log would break
df = df[(df['firms_2022'] > 0) & (df['pop_2023'] > 0) & (df['firms_1989'] > 0)]

# Log-transform outcome and population + 1989 firms
df['log_firms_2022'] = np.log(df['firms_2022'])
df['log_pop_2023'] = np.log(df['pop_2023'])
df['log_firms_1989'] = np.log(df['firms_1989'])

# Standardize income and education
scaler = StandardScaler()
df[['income_1989_scaled', 'pct_hs_1990_scaled']] = scaler.fit_transform(
    df[['income_1989_real_2023', 'pct_hs_1990']]
)

# Run the regression with persistence and 2020 density
model = smf.ols(
    formula='log_firms_2022 ~ church_persistence_index + church_density_2020 + income_1989_scaled + pct_hs_1990_scaled + log_pop_2023 + log_firms_1989 + C(State)',
    data=df
).fit(cov_type='HC3')

print(model.summary())


                            OLS Regression Results                            
Dep. Variable:         log_firms_2022   R-squared:                       0.963
Model:                            OLS   Adj. R-squared:                  0.963
Method:                 Least Squares   F-statistic:                     1571.
Date:                Wed, 20 Aug 2025   Prob (F-statistic):               0.00
Time:                        12:12:34   Log-Likelihood:                -565.33
No. Observations:                2968   AIC:                             1239.
Df Residuals:                    2914   BIC:                             1562.
Df Model:                          53                                         
Covariance Type:                  HC3                                         
                               coef    std err          z      P>|z|      [0.025      0.975]
--------------------------------------------------------------------------------------------
Intercept               